# 03 · Loss and resolution ablations

Rescores the single-seed ablation runs with exactly the protocol of `02_evaluate.ipynb`
(TTA, closing, validation-chosen threshold, test scored once), and compares each against
the base U-Net run **of the same seed** (adversarial weight 0.3, road weight 10, 256 input).

Labels come from the run name, since all ablation runs are plain RGB U-Nets. The 512 run is
built and fed at 512, and its output is area-resized to 256 before scoring.

## 1. Imports

In [ ]:
import csv, json, math, time, random
from pathlib import Path

import numpy as np
import cv2
import tensorflow as tf
import matplotlib.pyplot as plt

from sat2map import models, data, training, evaluation as ev, runs

print("TensorFlow", tf.__version__)
for g in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(g, True)
print("GPUs:", tf.config.list_physical_devices("GPU"))

from scipy.stats import wilcoxon

In [ ]:
DATA_ROOT = "../data/Dataset"   # folder holding train/ val/ test/
CKPT_ROOT = "../runs"           # searched recursively for generator_best.weights.h5
OUT       = Path("../eval")

# run folder names as written by 01_train.ipynb
BASE_RUN = "unet_res256_adv03_rw10_n2400_s96"
RUNS_TO_SCORE = [
    BASE_RUN,
    "unet_res256_adv00_rw10_n2400_s96",     # no adversarial term
    "unet_res256_adv10_rw10_n2400_s96",     # full adversarial term
    "unet_res256_adv03_rw00_n2400_s96",     # no road weighting
    "unet_res512_adv03_rw10_n2400_s96",     # 512 input
]
N_SAMPLES, BOOT, USE_TTA, USE_CLOSING = 8, 2000, True, True
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "samples").mkdir(exist_ok=True)

## 2. Checkpoints

In [ ]:
found = runs.find_checkpoints(CKPT_ROOT)
RUN_INFO = {}
for name in RUNS_TO_SCORE:
    if name not in found:
        print("MISSING  %s" % name); continue
    info = runs.parse(name)
    RUN_INFO[name] = dict(path=found[name], **info,
                          label="adv%.1f_rw%d_res%d" % (info["adv"], info["rw"], info["size"]))
if BASE_RUN not in RUN_INFO:
    raise SystemExit("The base run is required for the paired comparison.")
for n, d in RUN_INFO.items():
    print("%-36s %-22s %s" % (n, d["label"], d["path"]))

## 3. Data and target masks (always at 256)

In [ ]:
VAL_FILES  = data.file_list(DATA_ROOT, "val")
TEST_FILES = data.file_list(DATA_ROOT, "test")
DS = {256: (data.make_ds(VAL_FILES, 256), data.make_ds(TEST_FILES, 256))}
VAL_TAR, TEST_TAR = ev.target_masks(DS[256][0]), ev.target_masks(DS[256][1])

# figure tiles: spread across road density, road present
true_frac = TEST_TAR.reshape(TEST_TAR.shape[0], -1).mean(axis=1)
has_road = np.where(true_frac > 0)[0]
by_density = has_road[np.argsort(true_frac[has_road])]
SAMPLE_IDX = sorted(set(by_density[np.linspace(0, len(by_density) - 1, N_SAMPLES).astype(int)].tolist()))
print("target masks", VAL_TAR.shape, TEST_TAR.shape, "| sample tiles", SAMPLE_IDX)

## 4. Score every run

The full validation sweep is saved, not only its maximum, so precision and predicted road
density can be plotted against the threshold.

In [ ]:
rows, per_image_rows, sweep_rows, per_image_iou = [], [], [], {}

for name, d in RUN_INFO.items():
    size = d["size"]
    print("scoring %-36s (input %d)" % (name, size), flush=True)
    if size not in DS:
        DS[size] = (data.make_ds(VAL_FILES, size), data.make_ds(TEST_FILES, size))
    val_ds, test_ds = DS[size]

    gen = models.make_generator("unet", size)
    gen.load_weights(d["path"])
    val_score, _     = ev.road_score(gen, val_ds, tta=USE_TTA)
    test_score, kept = ev.road_score(gen, test_ds, tta=USE_TTA, keep_idx=set(SAMPLE_IDX))
    del gen; tf.keras.backend.clear_session()

    best_th, best_val, sweep = ev.sweep_threshold(val_score, VAL_TAR, ev.THRESHOLDS["rgb"], USE_CLOSING)
    sweep_rows += [dict(run=name, label=d["label"], thresh=th, **sv) for th, sv in sweep]
    per, tot = ev.collect(test_score, TEST_TAR, best_th, USE_CLOSING)
    s = ev.summarise(per, tot)
    sF = ev.summarise(*ev.collect(test_score, TEST_TAR, ev.FIXED_THRESH["rgb"], USE_CLOSING))

    rows.append(dict(run=name, label=d["label"], seed=d["seed"], size=size, adv=d["adv"],
                     rw=d["rw"], val_best_th=best_th, val_iou_perimg=best_val, **s,
                     fixed_th=ev.FIXED_THRESH["rgb"], iou_perimg_fixed=sF["iou_perimg"],
                     iou_pooled_fixed=sF["iou_pooled"]))
    per_image_iou[name] = np.array(per["iou"], float)
    for i, f in enumerate(TEST_FILES):
        per_image_rows.append(dict(run=name, label=d["label"], idx=i, file=Path(f).name,
                                   thresh=best_th, iou_buffered=per["iou"][i],
                                   iou_strict=per["strict"][i], precision=per["prec"][i],
                                   recall=per["rec"][i], pred_frac=per["frac"][i],
                                   true_frac=float(true_frac[i])))
    rdir = OUT / "samples" / name; rdir.mkdir(parents=True, exist_ok=True)
    for i, img in kept.items():
        cv2.imwrite(str(rdir / ("%03d.png" % i)), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    print("   th %.2f | val %.4f | test IoU %.4f pooled %.4f | prec %.4f rec %.4f | pred/true %.2f"
          % (best_th, best_val, s["iou_perimg"], s["iou_pooled"], s["prec_perimg"],
             s["rec_perimg"], s["density_ratio"]))

def write_csv(path, rows_):
    with open(path, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows_[0].keys()))
        w.writeheader(); w.writerows(rows_)

write_csv(OUT / "ablation_eval.csv", rows)
write_csv(OUT / "ablation_per_image.csv", per_image_rows)
write_csv(OUT / "ablation_val_sweep.csv", sweep_rows)

## 5. Sanity check

The base run is also scored by `02_evaluate.ipynb`. If its `final_eval.csv` is in `OUT`, the two
numbers must match; a mismatch means the two notebooks are not applying the same protocol.

In [ ]:
got = next(r["iou_perimg"] for r in rows if r["run"] == BASE_RUN)
ref_path = OUT / "final_eval.csv"
ref = None
if ref_path.exists():
    with open(ref_path) as fh:
        ref = next((float(r["iou_perimg"]) for r in csv.DictReader(fh) if r["run"] == BASE_RUN), None)
if ref is None:
    print("base run not found in %s; run 02_evaluate.ipynb first to enable this check" % ref_path)
else:
    print("base run test IoU: here %.6f, final_eval.csv %.6f" % (got, ref))
    print("MATCH" if abs(got - ref) < 1e-3 else
          "MISMATCH: check TTA, closing, threshold grid and dataset path")

## 6. Paired comparison against the base run and summary

In [ ]:
rng = np.random.default_rng(0)
base = per_image_iou[BASE_RUN]
hdr = "%-22s %9s %9s %9s %8s %11s" % ("label", "mean diff", "CI low", "CI high", "better%", "Wilcoxon p")
print(hdr); print("-" * len(hdr))
comp = []
for name, d in RUN_INFO.items():
    if name == BASE_RUN:
        continue
    b = ev.paired_bootstrap(per_image_iou[name], base, BOOT, rng)
    p = float(wilcoxon(b["diffs"]).pvalue)
    print("%-22s %9.4f %9.4f %9.4f %7.1f%% %11.2e" % (d["label"], b["mean_diff"], b["ci_low"],
                                                      b["ci_high"], 100 * b["frac_better"], p))
    comp.append(dict(run=name, label=d["label"], baseline=BASE_RUN, n_tiles=b["n_tiles"],
                     mean_diff=b["mean_diff"], ci_low=b["ci_low"], ci_high=b["ci_high"],
                     frac_better=b["frac_better"], wilcoxon_p=p,
                     excludes_zero=not (b["ci_low"] <= 0 <= b["ci_high"])))
write_csv(OUT / "ablation_paired.csv", comp)

print()
hdr = "%-22s %5s %8s %8s %8s %8s %8s %9s" % ("label", "th", "val IoU", "test IoU",
                                            "strict", "prec", "recall", "pred/true")
print(hdr); print("-" * len(hdr))
for r in rows:
    print("%-22s %5.2f %8.4f %8.4f %8.4f %8.4f %8.4f %9.2f" % (
        r["label"], r["val_best_th"], r["val_iou_perimg"], r["iou_perimg"], r["strict_perimg"],
        r["prec_perimg"], r["rec_perimg"], r["density_ratio"]))

## 7. Save the matching inputs and targets for the figure

In [ ]:
src_dir = OUT / "samples" / "_input"; src_dir.mkdir(parents=True, exist_ok=True)
for i, (src, tar) in enumerate(DS[256][1]):
    if i in SAMPLE_IDX:
        for kind, t in (("sat", src), ("target", tar)):
            img = np.clip((t[0].numpy() * 0.5 + 0.5) * 255, 0, 255).astype(np.uint8)
            cv2.imwrite(str(src_dir / ("%03d_%s.png" % (i, kind))), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
print("outputs in", OUT)